# Next-month spend

**You are here:** three regressors. Choose one on validation MAE, then score the test rows once.

Classification asks whether a customer places no paid order in July. Regression asks how many baht they spend that month.

`next_month_spend_thb` is 0 when `churned` is 1 and positive otherwise. Neither column is a feature.

Use the same split as the churn notebooks. Each model uses the two new columns, scaling, one-hot encoding, and `SelectKBest` with 5 features (`f_regression`). Do not use ROC-AUC here.

If a search is still running after a minute, stop it and set `n_jobs=-1`, or keep only the first value in each list.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    if (candidate / "app.py").is_file() and (candidate / "lib" / "schema.py").is_file():
        root = candidate
        break
    nested = candidate / "01-supervised-learning"
    if (nested / "app.py").is_file() and (nested / "lib" / "schema.py").is_file():
        root = nested
        break
if root is None:
    raise FileNotFoundError(
        "Start Jupyter from the repo folder or from 01-supervised-learning."
    )
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("cwd", here)
print("module", ROOT)


In [ ]:
import pandas as pd

customers = pd.read_csv(ROOT / "data" / "fresh_and_good_customers.csv")
print(customers.shape)
customers.head()


In [ ]:
from frame import features_and_target
from split import split_customers

train, validation, test = split_customers(customers)
print("train", len(train), "validation", len(validation), "test", len(test))
X_train, y_train = features_and_target(train)
X_val, y_val = features_and_target(validation)
X_test, y_test = features_and_target(test)
print("churned is a label, not a feature:", "churned" not in X_train.columns)


In [ ]:
from pipelines import fit_named
from schema import SPEND_TARGET

y_train_spend = train[SPEND_TARGET]
y_val_spend = validation[SPEND_TARGET]
y_test_spend = test[SPEND_TARGET]
ridge_search, ridge_seconds = fit_named("ridge", X_train, y_train_spend, "spend")
forest_search, forest_seconds = fit_named("forest", X_train, y_train_spend, "spend")
print("ridge and forest searches finished")
print("ridge seconds", round(ridge_seconds, 1))
print("forest seconds", round(forest_seconds, 1))


## Add the boosting model

Build `hgb_pipeline` with `make_candidate_pipeline` and `HistGradientBoostingRegressor(random_state=42)`. Use `task="spend"`.

`hgb_grid` searches `model__learning_rate` at `0.05` and `0.1`, and `model__max_depth` at `3` and `6`.

The next cell fits that grid on the train rows and prints the comparison. Expect ridge validation MAE near 1045 THB. The forest and the boosting model are a little higher. Pick the lowest validation MAE.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

from pipelines import make_candidate_pipeline

hgb_pipeline = None
hgb_grid = None


In [ ]:
import time

from sklearn.model_selection import GridSearchCV

from pipelines import format_comparison, score_row, spend_folds

if hgb_pipeline is None or hgb_grid is None:
    raise NotImplementedError("Build the boosting pipeline and its grid.")
started = time.perf_counter()
hgb_search = GridSearchCV(
    hgb_pipeline,
    hgb_grid,
    scoring="neg_mean_absolute_error",
    cv=spend_folds(),
    refit=True,
    n_jobs=1,
)
hgb_search.fit(X_train, y_train_spend)
hgb_seconds = time.perf_counter() - started
rows = [
    score_row("ridge", ridge_search, ridge_seconds, X_val, y_val_spend, "spend"),
    score_row("forest", forest_search, forest_seconds, X_val, y_val_spend, "spend"),
    score_row("hist_gradient", hgb_search, hgb_seconds, X_val, y_val_spend, "spend"),
]
print(format_comparison(rows, "spend"))


### Exercise

`chosen_spend` is `"ridge"`, `"forest"`, or `"hist_gradient"`. Set it to the name with the lowest validation MAE. Expect `ridge`.


In [ ]:
chosen_spend = None


In [ ]:
if chosen_spend not in {"ridge", "forest", "hist_gradient"}:
    raise NotImplementedError("Set chosen_spend to ridge, forest, or hist_gradient.")
fitted = {"ridge": ridge_search, "forest": forest_search, "hist_gradient": hgb_search}
winner = fitted[chosen_spend].best_estimator_
print("chosen", chosen_spend)


### Exercise. Test set, once

Assign test MAE and test RMSE in THB for `winner`. Use `spend_scores` on the test rows. Predictions are clipped at 0 THB. Run this once.


In [ ]:
from spend import spend_scores

test_mae = None
test_rmse = None


In [ ]:
if test_mae is None or test_rmse is None:
    raise NotImplementedError("Assign test MAE and test RMSE. Run the test cell once.")
print("test MAE", round(float(test_mae), 2))
print("test RMSE", round(float(test_rmse), 2))


In [ ]:
from persist import save_spend_bundle

destination = ROOT / "models" / "spend_bundle.joblib"
save_spend_bundle(destination, winner)
print("saved", destination)
print("Load only this file or one your instructor hands you.")
print("A joblib file can run code when it is loaded.")


## Checkpoint

The form reads both bundles. A churn file in the spend slot is rejected. Next, open `app.py`.
